# XGBoost Tutorial (Colab — Kaggle 다운로드 버전)

Kaggle의 [XGBoost Tutorial](https://www.kaggle.com/code/dansbecker/xgboost) (by Dan Becker) 을 **Google Colab에서 실행**하는 노트북입니다.

- 실행 시 **Kaggle API로 대회 데이터를 직접 다운로드**합니다 ([House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) 의 `train.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 원본은 2018년 코드라 일부 구식 API를 최신 라이브러리에 맞게 갱신했습니다:
  - `sklearn.preprocessing.Imputer` → `sklearn.impute.SimpleImputer`
  - `DataFrame.as_matrix()` → `.to_numpy()`
  - `fit(early_stopping_rounds=...)` → 생성자 인자 `XGBRegressor(early_stopping_rounds=...)` (xgboost 2.x+)

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [1]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "xgboost", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

라이브러리 준비 완료


In [2]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

Kaggle 자격증명 설정 완료 (내장 토큰)


## 1. Kaggle에서 House Prices 데이터 다운로드

In [3]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

# 대회 데이터 다운로드 + 압축 해제
api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

100%|██████████| 199k/199k [00:00<00:00, 11.0MB/s]


다운로드 완료: ['sample_submission.csv', 'test.csv', 'train.csv']


## 2. 데이터 준비 · XGBoost 학습 & 제출 파일 생성
train/test 로드 → **수치형 특징만 선택**(`select_dtypes`) → 결측치 **`SimpleImputer`** 대치 → train/val 분할 → **`XGBRegressor`**(`n_estimators=1000`, `learning_rate=0.05`, `early_stopping_rounds=50`) 학습 → test 예측 → `submission.csv`(`Id, SalePrice`) 저장.

In [5]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
train=pd.read_csv("/content/train.csv")
test=pd.read_csv("/content/test.csv")
y=train['SalePrice']
X=train.drop(["SalePrice","Id"],axis=1).select_dtypes(exclude=["object"])
X_test=test[X.columns]
imputer=SimpleImputer()
X_imputed=imputer.fit_transform(X)
X_test_imputed=imputer.transform(X_test)
X_tr,X_val,y_tr,y_val=train_test_split(X_imputed,y,test_size=0.2,random_state=42)
model=XGBRegressor(n_estimators=1000,learning_rate=0.05,early_stopping_rounds=50)
model.fit(X_tr,y_tr,eval_set=[(X_val,y_val)],verbose=False)
preds = model.predict(X_test_imputed)
pd.DataFrame({'Id': test.Id, 'SalePrice': preds}).to_csv("/content/submission.csv", index=False)

## 3. 제출 파일 내려받기

In [8]:
try:
    from google.colab import files
    files.download("/content/submission.csv")
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", "/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[House Prices 제출 페이지](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit)**

- 대회 홈: https://www.kaggle.com/c/house-prices-advanced-regression-techniques
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.